<a href="https://colab.research.google.com/github/lubbad-aljazeera/paid-activity-data-pipeline/blob/main/GoogleAds_Extended.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from google.cloud import storage, bigquery
from google.colab import auth
from datetime import datetime
import re
import os

# 1. AUTHENTICATION & CONFIGURATION
auth.authenticate_user()
project_id = "ajgc-dig-jwc-prod-jawacloud-01"
dataset_id = "aj360"
table_id = "google_ads_performance"
bucket_name = "aj360_data"
source_file = "GOOGLE_ADS/google_ads_performance_1Feb-7Mar2026.csv"

bq_client = bigquery.Client(project=project_id)
table_ref = f"{project_id}.{dataset_id}.{table_id}"

# ---------------------------------------------------------
# STEP 1: MONTHLY BACKUP MECHANISM
# ---------------------------------------------------------
backup_suffix = datetime.now().strftime('%Y_%m_%d').lower()
backup_table_id = f"{table_id}_backup_{backup_suffix}"
backup_ref = f"{project_id}.{dataset_id}.{backup_table_id}"

try:
    bq_client.get_table(table_ref)
    copy_job = bq_client.copy_table(
        table_ref,
        backup_ref,
        job_config=bigquery.CopyJobConfig(write_disposition="WRITE_TRUNCATE")
    )
    copy_job.result()
    print(f"✅ Backup created: {backup_table_id}")
except Exception as e:
    print(f"⚠️ Backup skipped: {e}")

# ---------------------------------------------------------
# STEP 2: LOAD LOOKUP TABLE
# ---------------------------------------------------------
try:
    lookup_query = f"SELECT term, show_title_en FROM `{project_id}.{dataset_id}.paid_ads_shows_lookup`"
    lookup_df = bq_client.query(lookup_query).to_dataframe()
    term_lookup = dict(zip(lookup_df['term'].str.lower(), lookup_df['show_title_en']))
except Exception as e:
    print(f"⚠️ Lookup failed: {e}")
    term_lookup = {}

# ---------------------------------------------------------
# STEP 3: DOWNLOAD AND CLEAN CSV
# ---------------------------------------------------------
local_csv = "/tmp/source.csv"
storage.Client().bucket(bucket_name).blob(source_file).download_to_filename(local_csv)

# Resilient reading with 'skip_leading_rows' handled via skiprows
df = pd.read_csv(local_csv, skiprows=2, on_bad_lines='skip', engine='python')

# Standardize column names early
df.columns = [re.sub(r'[^a-zA-Z0-9_]', '', col.lower().replace(' ', '_').replace('/', '_')) for col in df.columns]

# ---------------------------------------------------------
# STEP 4: SHOW TITLE EXTRACTION & DATA CLEANING
# ---------------------------------------------------------
def extract_show_title(row):
    # Search logic across available name columns
    search_text = f"{row.get('ad_name', '')}_{row.get('ad_group', '')}_{row.get('campaign', '')}"
    parts = re.split(r'[_ \-]', str(search_text).lower())
    for part in parts:
        if part in term_lookup:
            if term_lookup[part] != "AJ360 Branding":
                return term_lookup[part]
    return "AJ360 Branding"

df['show_title'] = df.apply(extract_show_title, axis=1)

# Thorough cleaning: strip whitespace, quotes, and handle BQ-unfriendly strings
df = df.apply(lambda x: x.str.strip().replace({r'\r': '', r'\n': '', '"': ''}, regex=True) if x.dtype == "object" else x)
df = df.replace(['nan', 'NaN', 'None', '--'], np.nan)

# Clean numeric columns
numeric_patterns = ['clicks', 'impr', 'conversions', 'viewthrough_conv', 'cost', 'ctr', 'conv_rate']
for col in df.columns:
    if any(p in col for p in numeric_patterns):
        df[col] = df[col].astype(str).str.replace(r'[,%]', '', regex=True)
        df[col] = pd.to_numeric(df[col], errors='coerce')

# ---------------------------------------------------------
# STEP 5: SCHEMA MONITORING & ALIGNMENT (The Automated Check)
# ---------------------------------------------------------
target_table = bq_client.get_table(table_ref)
target_schema = target_table.schema
schema_col_names = [field.name for field in target_schema]

# DETECT NEW COLUMNS
incoming_cols = set(df.columns)
table_cols = set(schema_col_names)
new_columns = incoming_cols - table_cols

if new_columns:
    print(f"🚨 ALERT: Detected {len(new_columns)} new columns in CSV not present in BigQuery:")
    for col in new_columns:
        print(f"   - {col}")
    print("💡 Action: Update your BigQuery table schema if you want to capture this data.")

# FORCE ALIGNMENT
for col in schema_col_names:
    if col not in df.columns:
        df[col] = None  # Add missing table columns as null

# Final reorder and drop extra columns (fixes the 400 error)
df = df[schema_col_names]

# ---------------------------------------------------------
# STEP 6: PRODUCTION LOAD
# ---------------------------------------------------------
job_config = bigquery.LoadJobConfig(
    schema=target_schema,
    write_disposition="WRITE_APPEND",
    source_format="CSV",
    allow_quoted_newlines=True,
    allow_jagged_rows=True
)

job = bq_client.load_table_from_dataframe(df, table_ref, job_config=job_config)
job.result()

print(f"✅ Data processed. Appended {len(df)} rows to {table_id}.")

✅ Backup created: google_ads_performance_backup_2026_03_08


/tmp/ipykernel_2175/3608071553.py:79: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace(['nan', 'NaN', 'None', '--'], np.nan)


🚨 ALERT: Detected 1 new columns in CSV not present in BigQuery:
   - adcomposed_image_id
💡 Action: Update your BigQuery table schema if you want to capture this data.
✅ Data processed. Appended 1741 rows to google_ads_performance.


In [2]:
import pandas as pd
import numpy as np
from google.cloud import storage, bigquery
from google.colab import auth
from datetime import datetime
import re
import os

# 1. AUTHENTICATION & CONFIGURATION
auth.authenticate_user()
project_id = "ajgc-dig-jwc-prod-jawacloud-01"
dataset_id = "aj360"
table_id = "googls_ads_country_device"  # Keeping your spelling
bucket_name = "aj360_data"
# Separating prefix from filename for cleaner GCS handling
source_file_path = "GOOGLE_ADS/google_ads_country_5-7Mar2026.csv"

bq_client = bigquery.Client(project=project_id)
table_ref = f"{project_id}.{dataset_id}.{table_id}"

# ---------------------------------------------------------
# STEP 1: MONTHLY BACKUP MECHANISM
# ---------------------------------------------------------
backup_suffix = datetime.now().strftime('%Y_%m_%d').lower()
backup_table_id = f"{table_id}_backup_{backup_suffix}"
backup_ref = f"{project_id}.{dataset_id}.{backup_table_id}"

try:
    # Check if main table exists
    bq_client.get_table(table_ref)
    # Monthly backup (Truncate ensures it refreshes if run multiple times in one month)
    copy_job = bq_client.copy_table(
        table_ref,
        backup_ref,
        job_config=bigquery.CopyJobConfig(write_disposition="WRITE_TRUNCATE")
    )
    copy_job.result()
    print(f"✅ Backup created/updated: {backup_table_id}")
except Exception as e:
    print(f"⚠️ Backup skipped: {e}")

# ---------------------------------------------------------
# STEP 2: DOWNLOAD AND RESILIENT CSV READING
# ---------------------------------------------------------
local_csv = "/tmp/source_country.csv"
storage.Client(project=project_id).bucket(bucket_name).blob(source_file_path).download_to_filename(local_csv)

# Use on_bad_lines='skip' as per the resilient loading requirement
df = pd.read_csv(local_csv, skiprows=2, on_bad_lines='skip', engine='python')

# ---------------------------------------------------------
# STEP 3: DATA CLEANING & TYPE CASTING
# ---------------------------------------------------------
# Standardize column names
df.columns = [re.sub(r'[^a-zA-Z0-9_]', '', col.lower().replace(' ', '_').replace('/', '_')) for col in df.columns]

# Comprehensive Cleaning: Remove quotes, newlines, and 'nan' strings
df = df.apply(lambda x: x.str.strip().replace({r'\r': '', r'\n': '', '"': ''}, regex=True) if x.dtype == "object" else x)
df = df.replace(['nan', 'NaN', 'None', '--'], np.nan)

# Clean numeric columns (Commas/Percentages)
numeric_indicators = ['clicks', 'impr', 'cost', 'conversions', 'ctr', 'conv_rate', 'interactions']
for col in df.columns:
    if any(ind in col for ind in numeric_indicators):
        df[col] = df[col].astype(str).str.replace(r'[,%]', '', regex=True)
        df[col] = pd.to_numeric(df[col], errors='coerce')

# ---------------------------------------------------------
# STEP 4: SCHEMA MONITORING & ALIGNMENT
# ---------------------------------------------------------
target_table = bq_client.get_table(table_ref)
target_schema = target_table.schema
schema_col_names = [field.name for field in target_schema]

# ALERT ON NEW COLUMNS
new_cols = set(df.columns) - set(schema_col_names)
if new_cols:
    print(f"🚨 ALERT: CSV contains columns not in BigQuery: {list(new_cols)}")

# ENSURE ALIGNMENT: Add missing columns, drop extra ones
for col in schema_col_names:
    if col not in df.columns:
        df[col] = None

# Reorder to match BQ precisely
df = df[schema_col_names]

# ---------------------------------------------------------
# STEP 5: PRODUCTION LOAD (WRITE_APPEND)
# ---------------------------------------------------------
job_config = bigquery.LoadJobConfig(
    schema=target_schema,
    write_disposition="WRITE_APPEND", # Switched to Append to complement Backup strategy
    source_format="CSV",
    allow_quoted_newlines=True,
    allow_jagged_rows=True,
    max_bad_records=10
)

# Load from dataframe to maintain local data type integrity
job = bq_client.load_table_from_dataframe(df, table_ref, job_config=job_config)
job.result()

print(f"✅ Successfully appended {len(df)} rows to {table_id}.")
display(df.head())

✅ Backup created/updated: googls_ads_country_device_backup_2026_03_08
✅ Successfully appended 480 rows to googls_ads_country_device.


,day,campaign,ad_group,campaign_type,campaign_subtype,country_territory_user_location,device,currency_code,campaign_id,ad_group_id,...,impr,viewthrough_conv,interactions,ctr,avg_cpc,cost,conversions,cost___conv,conv_rate,interaction_rate
0,2026-03-07,AJ360_FEB26_Qal-Alhakim_GoogleDemandGen_GCC_Tr...,AJ360_FEB26_Qal-Alhakim_GoogleDemandGen__Traff...,Demand Gen,All features,Saudi Arabia,Mobile phones,USD,23574391383,193746616495,...,124831,8,7567,1.82,0.01,31.45,0.0,0.00,0.00,6.06%
1,2026-03-06,AJ360_FEB26_Qal-Alhakim_GoogleDemandGen_GCC_Tr...,AJ360_FEB26_Qal-Alhakim_GoogleDemandGen__Traff...,Demand Gen,All features,Saudi Arabia,Mobile phones,USD,23574391383,193746616495,...,106755,17,6198,1.68,0.02,27.09,1.0,27.09,0.02,5.81%
2,2026-03-07,AJ360_FEB26_Qal-Alhakim_GoogleDemandGen_Levant...,AJ360_FEB26_Qal-Alhakim_GoogleDemandGen__Traff...,Demand Gen,All features,Jordan,Mobile phones,USD,23584371994,193746616695,...,49313,17,3644,2.39,0.02,27.24,0.0,0.00,0.00,7.39%
3,2026-03-07,AJ360_FEB26_NoName_GoogleDemandGen_GCC_Traffic,AJ360_FEB26_NoName_GoogleDemandGen__Traffic_GC...,Demand Gen,All features,Oman,Mobile phones,USD,23572935002,192063037094,...,95254,9,9779,1.22,0.05,53.95,0.0,0.00,0.00,10.27%
4,2026-03-06,AJ360_FEB26_Qal-Alhakim_GoogleDemandGen_Levant...,AJ360_FEB26_Qal-Alhakim_GoogleDemandGen__Traff...,Demand Gen,All features,Jordan,Mobile phones,USD,23584371994,193746616695,...,48056,16,2947,2.35,0.02,24.53,0.0,0.00,0.00,6.13%


In [3]:
import pandas as pd
import numpy as np
from google.cloud import storage, bigquery
from google.colab import auth
from datetime import datetime
import re
import os

# 1. AUTHENTICATION & CONFIGURATION
auth.authenticate_user()
project_id = "ajgc-dig-jwc-prod-jawacloud-01"
dataset_id = "aj360"
table_id = "googls_ads_age_gender"
bucket_name = "aj360_data"
# Fixed the source path logic (removing redundant bucket name)
source_file_path = "GOOGLE_ADS/google_ads_age_gender_5-7Mar2026.csv"

bq_client = bigquery.Client(project=project_id)
table_ref = f"{project_id}.{dataset_id}.{table_id}"

# ---------------------------------------------------------
# STEP 1: MONTHLY BACKUP MECHANISM
# ---------------------------------------------------------
backup_suffix = datetime.now().strftime('%Y_%m_%d').lower()
backup_table_id = f"{table_id}_backup_{backup_suffix}"
backup_ref = f"{project_id}.{dataset_id}.{backup_table_id}"

try:
    bq_client.get_table(table_ref)
    # WRITE_TRUNCATE ensures the monthly backup is a fresh snapshot
    copy_job = bq_client.copy_table(
        table_ref,
        backup_ref,
        job_config=bigquery.CopyJobConfig(write_disposition="WRITE_TRUNCATE")
    )
    copy_job.result()
    print(f"✅ Backup created/updated: {backup_table_id}")
except Exception as e:
    print(f"⚠️ Backup skipped: {e}")

# ---------------------------------------------------------
# STEP 2: RESILIENT CSV READING
# ---------------------------------------------------------
local_csv = "/tmp/source_age_gender.csv"
storage.Client(project=project_id).bucket(bucket_name).blob(source_file_path).download_to_filename(local_csv)

# Requirements: skip leading rows and skip bad lines for resilience
df = pd.read_csv(local_csv, skiprows=2, on_bad_lines='skip', engine='python')

# ---------------------------------------------------------
# STEP 3: DATA CLEANING & TYPE CASTING
# ---------------------------------------------------------
# Standardize column names (lower, no spaces, no special chars)
df.columns = [re.sub(r'[^a-zA-Z0-9_]', '', col.lower().replace(' ', '_').replace('/', '_')) for col in df.columns]

# Detailed Cleaning: Strip whitespace, remove quotes/newlines, handle 'nan'
df = df.apply(lambda x: x.str.strip().replace({r'\r': '', r'\n': '', '"': ''}, regex=True) if x.dtype == "object" else x)
df = df.replace(['nan', 'NaN', 'None', '--'], np.nan)

# Clean numeric columns (Commas/Percentages)
numeric_indicators = ['clicks', 'impr', 'cost', 'conversions', 'ctr', 'conv_rate', 'interactions']
for col in df.columns:
    if any(ind in col for ind in numeric_indicators):
        df[col] = df[col].astype(str).str.replace(r'[,%]', '', regex=True)
        df[col] = pd.to_numeric(df[col], errors='coerce')

# ---------------------------------------------------------
# STEP 4: SCHEMA ALIGNMENT & MONITORING
# ---------------------------------------------------------
target_table = bq_client.get_table(table_ref)
target_schema = target_table.schema
schema_col_names = [field.name for field in target_schema]

# DETECT NEW COLUMNS (Alerting only, doesn't break the load)
new_cols = set(df.columns) - set(schema_col_names)
if new_cols:
    print(f"🚨 ALERT: CSV contains columns not in BigQuery table: {list(new_cols)}")

# ENSURE ALIGNMENT: Fill missing columns with None, discard extra columns
for col in schema_col_names:
    if col not in df.columns:
        df[col] = None

# Final Reorder to match BQ schema precisely
df = df[schema_col_names]

# ---------------------------------------------------------
# STEP 5: PRODUCTION LOAD (WRITE_APPEND)
# ---------------------------------------------------------
job_config = bigquery.LoadJobConfig(
    schema=target_schema,
    write_disposition="WRITE_APPEND", # Continuous history strategy
    source_format="CSV",
    allow_quoted_newlines=True,
    allow_jagged_rows=True,
    max_bad_records=10
)

# Load from dataframe directly for better type mapping
job = bq_client.load_table_from_dataframe(df, table_ref, job_config=job_config)
job.result()

print(f"✅ Successfully appended {len(df)} rows to {table_id}.")
display(df.head())

# Cleanup
if os.path.exists(local_csv):
    os.remove(local_csv)

✅ Backup created/updated: googls_ads_age_gender_backup_2026_03_08
✅ Successfully appended 2868 rows to googls_ads_age_gender.


,day,campaign_id,campaign,ad_group_id,ad_group,campaign_type,campaign_subtype,device,age,gender,...,ctr,currency_code,avg_cpc,cost,conversions,viewthrough_conv,cost___conv,conv_rate,interactions,interaction_rate
0,2026-03-07,23574391383,AJ360_FEB26_Qal-Alhakim_GoogleDemandGen_GCC_Tr...,193746616495,AJ360_FEB26_Qal-Alhakim_GoogleDemandGen__Traff...,Demand Gen,All features,Mobile phones,25 - 34,Male,...,1.62,USD,0.02,7.59,0.0,3,0.0,0.0,1484,5.04%
1,2026-03-07,23574391383,AJ360_FEB26_Qal-Alhakim_GoogleDemandGen_GCC_Tr...,193746616495,AJ360_FEB26_Qal-Alhakim_GoogleDemandGen__Traff...,Demand Gen,All features,Mobile phones,35 - 44,Male,...,1.67,USD,0.02,8.02,0.0,2,0.0,0.0,1426,5.83%
2,2026-03-06,23574391383,AJ360_FEB26_Qal-Alhakim_GoogleDemandGen_GCC_Tr...,193746616495,AJ360_FEB26_Qal-Alhakim_GoogleDemandGen__Traff...,Demand Gen,All features,Mobile phones,25 - 34,Male,...,1.45,USD,0.02,6.52,0.0,3,0.0,0.0,1189,4.61%
3,2026-03-06,23574391383,AJ360_FEB26_Qal-Alhakim_GoogleDemandGen_GCC_Tr...,193746616495,AJ360_FEB26_Qal-Alhakim_GoogleDemandGen__Traff...,Demand Gen,All features,Mobile phones,35 - 44,Male,...,1.54,USD,0.02,7.62,0.0,9,0.0,0.0,1290,5.79%
4,2026-03-07,23574391383,AJ360_FEB26_Qal-Alhakim_GoogleDemandGen_GCC_Tr...,193746616495,AJ360_FEB26_Qal-Alhakim_GoogleDemandGen__Traff...,Demand Gen,All features,Mobile phones,25 - 34,Female,...,1.69,USD,0.01,4.22,0.0,3,0.0,0.0,1160,6.41%


In [4]:
import pandas as pd
import numpy as np
from google.cloud import storage, bigquery
from google.colab import auth
from datetime import datetime
import re
import os

# 1. AUTHENTICATION & CONFIGURATION
auth.authenticate_user()
project_id = "ajgc-dig-jwc-prod-jawacloud-01"
dataset_id = "aj360"
table_id = "googls_ads_conversions"
bucket_name = "aj360_data"
# Fixed the source path logic (removing redundant bucket name)
source_file_path = "GOOGLE_ADS/google_ads_conversions_5-7Mar2026.csv"

bq_client = bigquery.Client(project=project_id)
table_ref = f"{project_id}.{dataset_id}.{table_id}"

# ---------------------------------------------------------
# STEP 1: MONTHLY BACKUP MECHANISM
# ---------------------------------------------------------
backup_suffix = datetime.now().strftime('%Y_%m_%d').lower()
backup_table_id = f"{table_id}_backup_{backup_suffix}"
backup_ref = f"{project_id}.{dataset_id}.{backup_table_id}"

try:
    bq_client.get_table(table_ref)
    # WRITE_TRUNCATE ensures the monthly backup is a fresh snapshot
    copy_job = bq_client.copy_table(
        table_ref,
        backup_ref,
        job_config=bigquery.CopyJobConfig(write_disposition="WRITE_TRUNCATE")
    )
    copy_job.result()
    print(f"✅ Backup created/updated: {backup_table_id}")
except Exception as e:
    print(f"⚠️ Backup skipped: {e}")

# ---------------------------------------------------------
# STEP 2: RESILIENT CSV READING
# ---------------------------------------------------------
local_csv = "/tmp/source_conversions.csv"
storage.Client(project=project_id).bucket(bucket_name).blob(source_file_path).download_to_filename(local_csv)

# Requirements: skip leading rows and skip bad lines for resilience
df = pd.read_csv(local_csv, skiprows=2, on_bad_lines='skip', engine='python')

# ---------------------------------------------------------
# STEP 3: DATA CLEANING & TYPE CASTING
# ---------------------------------------------------------
# Standardize column names (lower, no spaces, no special chars)
df.columns = [re.sub(r'[^a-zA-Z0-9_]', '', col.lower().replace(' ', '_').replace('/', '_')) for col in df.columns]

# Detailed Cleaning: Strip whitespace, remove quotes/newlines, handle 'nan'
df = df.apply(lambda x: x.str.strip().replace({r'\r': '', r'\n': '', '"': ''}, regex=True) if x.dtype == "object" else x)
df = df.replace(['nan', 'NaN', 'None', '--'], np.nan)

# Clean numeric columns (Commas/Percentages)
numeric_indicators = ['clicks', 'impr', 'cost', 'conversions', 'ctr', 'conv_rate', 'interactions']
for col in df.columns:
    if any(ind in col for ind in numeric_indicators):
        df[col] = df[col].astype(str).str.replace(r'[,%]', '', regex=True)
        df[col] = pd.to_numeric(df[col], errors='coerce')

# ---------------------------------------------------------
# STEP 4: SCHEMA ALIGNMENT & MONITORING
# ---------------------------------------------------------
target_table = bq_client.get_table(table_ref)
target_schema = target_table.schema
schema_col_names = [field.name for field in target_schema]

# DETECT NEW COLUMNS (Alerting only, doesn't break the load)
new_cols = set(df.columns) - set(schema_col_names)
if new_cols:
    print(f"🚨 ALERT: CSV contains columns not in BigQuery table: {list(new_cols)}")

# ENSURE ALIGNMENT: Fill missing columns with None, discard extra columns
for col in schema_col_names:
    if col not in df.columns:
        df[col] = None

# Final Reorder to match BQ schema precisely
df = df[schema_col_names]

# ---------------------------------------------------------
# STEP 5: PRODUCTION LOAD (WRITE_APPEND)
# ---------------------------------------------------------
job_config = bigquery.LoadJobConfig(
    schema=target_schema,
    write_disposition="WRITE_APPEND", # Continuous history strategy
    source_format="CSV",
    allow_quoted_newlines=True,
    allow_jagged_rows=True,
    max_bad_records=10
)

# Load from dataframe directly for better type mapping
job = bq_client.load_table_from_dataframe(df, table_ref, job_config=job_config)
job.result()

print(f"✅ Successfully appended {len(df)} rows to {table_id}.")
display(df.head())

# Cleanup
if os.path.exists(local_csv):
    os.remove(local_csv)

✅ Backup created/updated: googls_ads_conversions_backup_2026_03_08


/tmp/ipykernel_156/1658589038.py:58: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace(['nan', 'NaN', 'None', '--'], np.nan)


🚨 ALERT: CSV contains columns not in BigQuery table: ['adcomposed_image_id']
✅ Successfully appended 141 rows to googls_ads_conversions.


,day,campaign_id,campaign,ad_group_id,ad_group,ad_id,ad_state,ad_type,final_url,beacon_urls,...,mobile_final_url,tracking_template,final_url_suffix,custom_parameter,conversion_action,campaign_type,device,ad_final_url,conversions,viewthrough_conv
0,2026-03-05,23577993117,AJ360_Feb26_AJ360_GoogleSearch__Awareness_GCC,196519063627,AJ360_Branded,797845233562,Enabled,Responsive search ad,https://www.aljazeera360.com/section/Ramadan?u...,NaN,...,NaN,NaN,NaN,NaN,Al Jazeera 360 - الجزيرة 360 (Android) install...,Search,Mobile phones,https://www.aljazeera360.com/section/Ramadan?u...,5.0,0
1,2026-03-06,23577993117,AJ360_Feb26_AJ360_GoogleSearch__Awareness_GCC,196519063627,AJ360_Branded,797845233562,Enabled,Responsive search ad,https://www.aljazeera360.com/section/Ramadan?u...,NaN,...,NaN,NaN,NaN,NaN,Al Jazeera 360 - الجزيرة 360 (Android) install...,Search,Mobile phones,https://www.aljazeera360.com/section/Ramadan?u...,3.0,0
2,2026-03-07,23577993117,AJ360_Feb26_AJ360_GoogleSearch__Awareness_GCC,196519063627,AJ360_Branded,797845233562,Enabled,Responsive search ad,https://www.aljazeera360.com/section/Ramadan?u...,NaN,...,NaN,NaN,NaN,NaN,Al Jazeera 360 - الجزيرة 360 (Android) install...,Search,Mobile phones,https://www.aljazeera360.com/section/Ramadan?u...,2.0,0
3,2026-03-05,23577993243,AJ360_Feb26_AJ360_GoogleSearch__Awareness_Levant,196519063827,AJ360_Branded,797845233568,Enabled,Responsive search ad,https://www.aljazeera360.com/section/Ramadan?u...,NaN,...,NaN,NaN,NaN,NaN,Al Jazeera 360 - الجزيرة 360 (Android) install...,Search,Mobile phones,https://www.aljazeera360.com/section/Ramadan?u...,2.0,0
4,2026-03-06,23577993243,AJ360_Feb26_AJ360_GoogleSearch__Awareness_Levant,196519063827,AJ360_Branded,797845233568,Enabled,Responsive search ad,https://www.aljazeera360.com/section/Ramadan?u...,NaN,...,NaN,NaN,NaN,NaN,Al Jazeera 360 - الجزيرة 360 (Android) install...,Search,Mobile phones,https://www.aljazeera360.com/section/Ramadan?u...,1.0,0


In [5]:
# Extract and list unique terms from Ad Name, Ad Set Name, and Campaign Name for debugging

extracted_terms = set()
df.head
if df is not None:
    # Extract terms from 'Ad name' (parts 1 and 2 after splitting by _ and -)
    if 'ad_name' in df.columns:
        for ad_name in df['ad_name'].dropna():
            ad_name_parts = re.split(r'[_ -]', ad_name)
            if len(ad_name_parts) > 0:
                extracted_terms.add(ad_name_parts[0]) # part1
            if len(ad_name_parts) > 1:
                extracted_terms.add(ad_name_parts[1]) # part2
            if len(ad_name_parts) > 2:
                extracted_terms.add(ad_name_parts[2]) # part2

    # Extract terms from 'Ad Set Name' (part 3 after splitting by _)
    if 'ad_group' in df.columns:
        for ad_set_name in df['ad_group'].dropna():
            ad_set_name_parts = ad_set_name.split('_')
            if len(ad_set_name_parts) > 0:
                extracted_terms.add(ad_set_name_parts[0]) # part1
            if len(ad_set_name_parts) > 1:
                extracted_terms.add(ad_set_name_parts[1]) # part2
            if len(ad_set_name_parts) > 2:
                extracted_terms.add(ad_set_name_parts[2]) # part3

    # Extract terms from 'Campaign Name' (part 3 after splitting by _)
    if 'campaign_name' in df.columns:
        for campaign_name in df['campaign_name'].dropna():
            campaign_name_parts = campaign_name.split('_')
            if len(campaign_name_parts) > 0:
                extracted_terms.add(campaign_name_parts[0]) # part3
            if len(campaign_name_parts) > 1:
                extracted_terms.add(campaign_name_parts[1]) # part3
            if len(campaign_name_parts) > 2:
                extracted_terms.add(campaign_name_parts[2]) # part3

# Separate matched and unmatched terms
matched_terms = sorted([term for term in extracted_terms if term in term_lookup])
unmatched_terms = sorted([term for term in extracted_terms if term not in term_lookup])

# Print the unique and distinct extracted terms
print("Unique and distinct terms extracted for lookup matching:")
print("--- Matched Terms ---")
if matched_terms:
    for term in matched_terms:
        print(term)
else:
    print("No terms matched with the lookup table.")

print("\n--- Unmatched Terms ---")
if unmatched_terms:
    for term in unmatched_terms:
        print(term)
else:
    print("All extracted terms were matched with the lookup table.")

Unique and distinct terms extracted for lookup matching:
--- Matched Terms ---
No terms matched with the lookup table.

--- Unmatched Terms ---
AJ360
BRANDED
Branded
Episode6
Episode8
FEB26
NoName
Qal-Alhakim
QalAlhakim
Rifqan
RifqanS2
